In [1]:
from transformers import (
  T5ForConditionalGeneration, 
  T5Tokenizer, Trainer, TrainingArguments
)
import pandas as pd 

In [2]:
train_df = pd.read_csv("Data/samsum-train.csv")
train_df.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [3]:
train_df.shape

(14732, 3)

In [4]:
test_df = pd.read_csv("Data/samsum-test.csv")
test_df.shape

(819, 3)

In [5]:
validation_df = pd.read_csv("Data/samsum-validation.csv")
validation_df.shape

(818, 3)

In [6]:
# taking a sample
train_df = train_df.sample(n = 6000, random_state = 42).reset_index(drop = True)
validation_df = validation_df.sample(n = 500 , random_state = 42).reset_index(drop = True)
test_df = test_df.sample(n = 500 , random_state = 42).reset_index(drop = True)

In [7]:
train_df.head()

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."


In [8]:
print(train_df['dialogue'][0])

Violet: hi! i came across this Austin's article and i thought that you might find it interesting
Violet: <file_other>
Claire: Hi! :) Thanks, but I've already read it. :)
Claire: But thanks for thinking about me :)


### Data Processing

In [9]:
import re 
def clean_text(text):
    text = re.sub(r'\r\n', ' ' , text)
    text = re.sub(r'\s+' , ' ' , text)
    text = re.sub(r'<.*?>' , ' ' , text)
    text = text.strip().lower()
    return text 

In [10]:
train_df['dialogue'] = train_df['dialogue'].apply(clean_text)
train_df['summary'] = train_df['summary'].apply(clean_text)

In [11]:
test_df['dialogue'] = test_df['dialogue'].apply(clean_text)
test_df['summary'] = test_df['summary'].apply(clean_text)

In [12]:
validation_df['dialogue'] = validation_df['dialogue'].apply(clean_text)
validation_df['summary'] = validation_df['summary'].apply(clean_text)

### Making dataset and datadict

In [13]:
from datasets import Dataset , DatasetDict

In [14]:
dataset = DatasetDict({
    'train' : Dataset.from_pandas(train_df),
    'test' : Dataset.from_pandas(test_df),
    'valid' : Dataset.from_pandas(validation_df)
})
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 6000
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 500
    })
    valid: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 500
    })
})

### Tokenization

In [15]:
model_checkpoint = 't5-small'
tokenizer = T5Tokenizer.from_pretrained(
    model_checkpoint,
    legacy = False
)

In [16]:
def tokenization(examples):
    # extract the  dialogue and summary
    dialogues = examples['dialogue']
    summaries = examples['summary']

    # tokenize diagues 
    inputs =  tokenizer(
        dialogues, 
        truncation = True,
        padding = 'max_length', 
        max_length = 512
    )
    targets = tokenizer(
        summaries, 
        truncation = True,
        padding = 'max_length', 
        max_length = 128
    )
    # get the labels
    labels = targets['input_ids']
    inputs['labels'] = labels
    return inputs

In [17]:
dataset['train'][0]['dialogue']

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [18]:
dataset['train'][0]['summary']

"violet sent claire austin's article."

In [19]:
tokenizer(dataset['train'][0]['dialogue'])

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [20]:
tokenized_dataset = dataset.map(tokenization, batched = True)

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

### Define the model

In [21]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [22]:
model = T5ForConditionalGeneration.from_pretrained(model_checkpoint)

In [24]:
from transformers import DataCollatorForSeq2Seq

In [25]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer = tokenizer,
    model = model
)

In [26]:
from transformers import Seq2SeqTrainingArguments

In [27]:
train_args = Seq2SeqTrainingArguments(
    output_dir = "./summarization-fine-tuning",
    num_train_epochs = 6,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    warmup_steps = 500,
    weight_decay = 0.01,
    logging_dir = './logs',
    logging_steps = 50,
    eval_steps = 50,
    save_steps = 500,
    eval_strategy = 'epoch'
)

In [29]:
trainer = Trainer(
    model = model,
    args = train_args,
    data_collator = data_collator,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['valid']
)

In [30]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.477500,0.427147
2,0.451000,0.410613
3,0.426800,0.405946
4,0.424300,0.401235
5,0.431600,0.400828
6,0.394600,0.400335


TrainOutput(global_step=4500, training_loss=0.7951507449679904, metrics={'train_runtime': 635.0871, 'train_samples_per_second': 56.685, 'train_steps_per_second': 7.086, 'total_flos': 4872304852992000.0, 'train_loss': 0.7951507449679904, 'epoch': 6.0})

### save the model

In [31]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\special_tokens_map.json',
 './saved_summary_model\\spiece.model',
 './saved_summary_model\\added_tokens.json')

In [33]:
device = model.device

In [34]:
def summerize_dialogue(text):
    text = clean_text(text)
    inputs = tokenizer(
        text,
        truncation = True,
        padding = 'max_length', 
        max_length = 512,
        return_tensors = 'pt'
    )

    inputs = {key : val.to(device) for key , val in inputs.items()}
    outputs = model.generate(
        inputs = inputs['input_ids'],
        max_length = 128,
        num_beams = 4,
        early_stopping = True
    )
    summary = tokenizer.decode(outputs[0] , skip_special_tokens = True)
    return summary

In [40]:
dialogue = dataset['test'][1]['dialogue']
summary = dataset['test'][1]['summary']

In [41]:
dialogue

"javier: hey do you know any tattoo parlors over here with english speaking employees? judie: oh there's warsaw ink javier: the name sounds neat... have you had a tattoo done there? judie: nope but my gf has javier: got a pic? judie:   javier: wow that looks amazing javier: how much did she pay? judie: it was a 1000 javier: fuck javier: let me just get a tatttoo back in colombia then, thx"

In [42]:
summary

'javier was initially eager to have a tatoo done at warsaw ink but the price turned out to be too high. javier decided to have a tatoo done in colombia.'

In [43]:
print(f"Summary: {summerize_dialogue(dialogue)}")

Summary: judie and javier have a warsaw ink tattoo done in colombia. he will get a tatttoo back in colombia.
